In [4]:
import pandas as pd
import numpy as np
from datatools import get_index_K, get_price, get_cyq, get_rzrq, get_moneyflow, get_technical
import lightgbm as lgb
import matplotlib.pyplot as plt
from cal_cne6_factor import calc_cne6_factors

### 取特征及样本

In [2]:
allstock = pd.read_csv('data/allstock.csv')
allstock_list = allstock[(allstock['list_date'] < 20260101) & ~allstock['ts_code'].str.contains('BJ')].ts_code.tolist()
delist = allstock[(allstock['list_date'] < 20260101) &(allstock['delist_date'] >= 20190101) & ~allstock['ts_code'].str.contains('BJ')].ts_code.tolist()

In [ ]:
check_sample = get_price(allstock_list, '2019-01-01', '2026-06-01',fields=['open'])
check_sample['5d_returns'] = check_sample.groupby('ts_code')['open'].shift(-5) / check_sample['open'] - 1
check_sample['10d_returns'] = check_sample.groupby('ts_code')['open'].shift(-10) / check_sample['open'] - 1
check_sample['15d_returns'] = check_sample.groupby('ts_code')['open'].shift(-15) / check_sample['open'] - 1
check_sample['20d_returns'] = check_sample.groupby('ts_code')['open'].shift(-20) / check_sample['open'] - 1
check_sample.rename(columns={'open':'next_open'}, inplace=True)
check_sample['trade_date'] = check_sample['trade_date'].shift(1)
check_sample = check_sample[check_sample['trade_date'].notna()]
check_sample = check_sample.sort_values(['trade_date','ts_code'])
check_sample.to_parquet('data/factor/all_sample.parquet', index=False)

In [14]:
de_sample = pd.read_parquet('data/factor/all_sample.parquet')
de_sample = de_sample[de_sample.ts_code.isin(delist)]

### 样本划分及预处理

In [3]:
factor1_de = calc_cne6_factors('2019-01-01', '2026-06-01', delist)

加载价格数据...
加载指数数据...
矩阵: 2033 交易日 × 221 只股票
计算 Daily std...
计算 Cumulative range...
计算 Short Term reversal...


c:\Users\User\Desktop\Quant\cal_cne6_factor.py:99: RuntimeWarning: All-NaN slice encountered
  factor_values = np.nanmax(window_rets, axis=0) - np.nanmin(window_rets, axis=0)


计算 Seasonality...
计算 Industry Momentum...
计算 BETA / Hist sigma / Historical alpha...
计算 Relative strength...
加载长期价格数据...
加载长期指数数据...
计算 Long term relative strength...
计算 Long term historical alpha...
加载基础数据...
加载财务数据...
加载TTM数据...
计算换手率因子...
计算市值因子...
计算估值/杠杆因子...
计算盈利能力...
计算财务质量因子...
计算应计项目...
计算投资质量...
加载分析师预测数据...
计算分析师预测因子...
组装结果...
done


In [5]:
cyq = get_cyq(delist, '2019-01-01', '2026-06-01')
rzrq = get_rzrq(delist, '2019-01-01', '2026-06-01')
monthflow = get_moneyflow(delist, '2019-01-01', '2026-06-01')
tech = get_technical(delist, '2019-01-01', '2026-06-01')

In [9]:
len(factor1_de)

396695

In [ ]:
all_sample = pd.read_parquet('data/factor/all_sample.parquet')
factor1 = pd.read_parquet('data/factor/cne6_factor.parquet')
factor2 = pd.read_parquet('data/factor/other_factor.parquet')
factor3 = pd.read_parquet('data/factor/tech_factor.parquet')

In [ ]:
sample = all_sample.query( 'trade_date >= "2023-01-01" and trade_date <= "2025-12-31"') ## default
sample = sample.merge(factor1, on=['ts_code', 'trade_date'], how='left').merge(
    factor2, on=['ts_code', 'trade_date'], how='left').merge(factor3, on=['ts_code', 'trade_date'], how='left')

In [19]:
bench = get_index_K(['000852.SH'], start_date='2023-01-01', end_date='2026-03-31', fields=['open'])  ## default
bench['bench_5d_return'] = bench['open'].shift(-5) / bench['open'] -1
bench['bench_10d_return'] = bench['open'].shift(-10) / bench['open'] -1
bench['bench_15d_return'] = bench['open'].shift(-15) / bench['open'] -1
bench['bench_20d_return'] = bench['open'].shift(-20) / bench['open'] -1
bench['trade_date'] = bench['trade_date'].shift(1)
bench = bench.query('trade_date >= "2023-01-01" and trade_date <= "2025-12-31"') ## default

In [20]:
sample = sample.merge(bench[['trade_date', 'bench_5d_return', 'bench_10d_return',
       'bench_15d_return', 'bench_20d_return']], on='trade_date', how='left')
sample['5d_excess_returns'] = sample['5d_returns'] - sample['bench_5d_return']
sample['10d_excess_returns'] = sample['10d_returns'] - sample['bench_10d_return']
sample['15d_excess_returns'] = sample['15d_returns'] - sample['bench_15d_return']
sample['20d_excess_returns'] = sample['20d_returns'] - sample['bench_20d_return']
sample.drop(columns=['5d_returns', '10d_returns', '15d_returns', '20d_returns', 'next_open', 'bench_5d_return', 'bench_10d_return', 'bench_15d_return', 'bench_20d_return'], inplace=True)
sample['Y_5d'] = sample.groupby('trade_date')['5d_excess_returns'].rank(pct=True)
sample['Y_10d'] = sample.groupby('trade_date')['10d_excess_returns'].rank(pct=True)
sample['Y_15d'] = sample.groupby('trade_date')['15d_excess_returns'].rank(pct=True)
sample['Y_20d'] = sample.groupby('trade_date')['20d_excess_returns'].rank(pct=True)

In [ ]:
def feats_preprocess(df, factor_cols, n_mad=5):
    # 按日期分组，对每个因子应用MAD
    def mad_clip(group):
        for col in factor_cols:
            median = group[col].median()
            mad = (group[col] - median).abs().median()
            # 防止mad为0（比如某因子一整列都一样）
            if mad < 1e-8:
                continue
            upper = median + n_mad * mad
            lower = median - n_mad * mad
            group[col] = group[col].clip(lower, upper)

            mean = group[col].mean()
            std = group[col].std()
            if std < 1e-8:
                group[col] = 0.0  # 波动为0的因子无信息，置0
            else:
                group[col] = (group[col] - mean) / std
        return group
    # 并行或滚动处理，根据数据量选择
    return df.groupby('trade_date').apply(mad_clip)

sample = feats_preprocess(sample, sample.columns[2:-8])
sample = sample.reset_index(level=0).reset_index(drop=True)
sample.to_parquet('data/factor/sample_oot.parquet', index=False)

C:\Users\User\AppData\Local\Temp\ipykernel_37208\622302902.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sample = sample.reset_index(level=0).reset_index(drop=True)


#### training

In [2]:
sample_train = pd.read_parquet('data/factor/sample_train.parquet')
sample_oot = pd.read_parquet('data/factor/sample_oot.parquet')
sample_oot = sample_oot.sample(frac=0.1, random_state=42)

In [5]:
lgb_params = {
    'objective': 'regression', # 虽然是Rank，但用regression配合分位数Y通常比lambdarank更稳定
    'metric': 'rmse',
    'learning_rate': 0.03,     # 学习率要低
    'max_depth': 4,            # 绝对不要超过5！金融数据浅层逻辑最稳
    'num_leaves': 10,          # 与max_depth匹配，限制复杂度
    'subsample': 0.7,          # 行采样（防共线性强的股票集群主导模型）
    'colsample_bytree': 0.6,   # 列采样（让150个因子都有机会参与，防某几个因子独占）
    'reg_alpha': 0.5,          # L1正则化
    'reg_lambda': 1.0,         # L2正则化
    'min_child_samples': 200,  # 叶子节点最小样本量（非常重要，防止模型去拟合极小众的极端妖股）
    'n_estimators': 1000,      # 设大一点，配合早停
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

In [14]:
# 假设 df 已经处理好了特征和Y
# 进一步从 train_val_df 中切出 20% 作为验证集 (按时间切，比如22年全年)
val_df = sample_train[sample_train['trade_date'] >= '2022-05-01']
train_df = sample_train[sample_train['trade_date'] < '2022-05-01']
feature_cols = sample_train.columns[2:-8]
lgb_train = lgb.Dataset(train_df[feature_cols], train_df['Y_20d'])
lgb_val = lgb.Dataset(val_df[feature_cols], val_df['Y_20d'])

model = lgb.train(
    lgb_params,
    lgb_train,
    valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
# 在 OOT 集(23年+)上进行日频预测
sample_oot['raw_score'] = model.predict(sample_oot[feature_cols])
# 对预测结果做截面标准化 (方便后续选股，虽然不影响Rank IC)
sample_oot['predict_score'] = sample_oot.groupby('trade_date')['raw_score'].transform(lambda x: (x - x.mean()) / x.std())
# 计算 OOT 期间的日度 Rank IC
daily_ic = sample_oot.groupby('trade_date').apply(lambda x: x['predict_score'].corr(x['Y_20d'], method='spearman'))
print(f"OOT 平均 Rank IC: {daily_ic.mean():.4f}")
print(f"OOT IC 胜率: {(daily_ic > 0).mean():.2%}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.283844
[200]	valid_0's rmse: 0.283355
[300]	valid_0's rmse: 0.28316
[400]	valid_0's rmse: 0.283016
[500]	valid_0's rmse: 0.282921
[600]	valid_0's rmse: 0.282871
[700]	valid_0's rmse: 0.282806
[800]	valid_0's rmse: 0.282766
[900]	valid_0's rmse: 0.282716
Early stopping, best iteration is:
[922]	valid_0's rmse: 0.282711
OOT 平均 Rank IC: 0.1231
OOT IC 胜率: 79.09%


In [ ]:
# ================= 1. 分层回测核心函数 =================
def layered_backtest(df, n_groups=10, cost=0.001):
    """
    df: 包含预测值和真实收益的 DataFrame，必须包含列：
        - 'date': 日期 (调仓日)
        - 'predict': 模型预测得分
        - 'forward_return': 真实收益率 (注意：必须是T+1到T+周期的收益，不能有前视偏差)
    n_groups: 分层数量，通常设为 5 或 10
    cost: 单边换仓成本 (例如千1.5，设为0.0015)
    """
    print(f"开始进行 {n_groups} 分层回测...")
    # 1. 截面分组 (按日期分组打标签)
    # pd.qcut 会将每天的数据按预测值等分为 n_groups 份
    # duplicates='drop' 防止预测值大量重复导致报错
    df['group'] = df.groupby('date')['predict'].transform(
        lambda x: pd.qcut(x.rank(method='first'), n_groups, labels=False, duplicates='drop')
    )
    # 2. 计算每组每日的等权平均收益
    # Group 0 是预测最差的组，Group n_groups-1 是预测最好的组
    group_ret = df.groupby(['date', 'group'])['forward_return'].mean().unstack()
    # 3. 处理换仓成本 (简化处理：假设每组每期全额换仓，扣除双边成本)
    # 实际上组内只有部分股票变动，这里扣除全额成本是非常保守的估计
    group_ret = group_ret - 2 * cost 
    # 4. 计算多空对冲收益 (做多最好组，做空最差组)
    group_ret['Long_Short'] = group_ret[n_groups - 1] - group_ret[0]
    # 5. 计算累计净值
    group_nav = (1 + group_ret).cumprod()
    return group_nav, group_ret
# ================= 2. 可视化函数 =================
def plot_layered_nav(group_nav):
    plt.rcParams['font.sans-serif'] = ['SimHei'] # 正常显示中文
    plt.rcParams['axes.unicode_minus'] = False
    fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [2, 1]})
    # 子图1：各组净值曲线 (扇形图)
    n_groups = len(group_nav.columns) - 1 # 减去 Long_Short 列
    colors = plt.cm.coolwarm(np.linspace(0, 1, n_groups))
    for i in range(n_groups):
        axes[0].plot(group_nav.index, group_nav[i], label=f'Group {i} (第{i+1}层)', color=colors[i], linewidth=1.5)
    axes[0].set_title('分层回测净值曲线 (扇形图)', fontsize=14)
    axes[0].set_ylabel('累计净值')
    axes[0].legend(loc='upper left')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    # 子图2：多空对冲净值曲线
    axes[1].plot(group_nav.index, group_nav['Long_Short'], label='多空对冲', color='green', linewidth=2)
    axes[1].axhline(y=1.0, color='black', linestyle='--', linewidth=0.8)
    axes[1].set_title('多空对冲净值曲线', fontsize=14)
    axes[1].set_xlabel('日期')
    axes[1].set_ylabel('累计净值')
    axes[1].legend(loc='upper left')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()
# ================= 3. 模拟运行 =================
# 假设你有 OOT 期间的数据框 oot_df，包含 'date', 'predict', 'forward_return'
# 如果没有，我生成一段模拟数据让你看效果
np.random.seed(42)
dates = pd.date_range('2023-01-01', '2024-01-01', freq='M')
tickers = [f'Stock_{i}' for i in range(500)]
mock_data = []
for d in dates:
    for t in tickers:
        # 模拟预测值
        pred = np.random.randn()
        # 模拟收益率：加入微弱的真实信号 + 噪声
        ret = 0.02 * pred + np.random.randn() * 0.1 
        mock_data.append({'date': d, 'ticker': t, 'predict': pred, 'forward_return': ret})
oot_df = pd.DataFrame(mock_data)
# --- 真正调用的一行代码 ---
group_nav, group_ret = layered_backtest(oot_df, n_groups=10, cost=0.0015)
# 打印各组最终权益和单调性
final_nav = group_nav.iloc[-1]
print("\n=== 各组最终权益 ===")
print(final_nav)
# 绘图
plot_layered_nav(group_nav)

### step 4

In [ ]:
# ================= 1. 设定组合参数 =================
TOTAL_CAPITAL = 50000      # 总资金量（假设5万）
TOP_N = 8                  # 持仓数量（5-10只较合适）
LAMBDA = 3.0               # 集中度系数
COST_BUY = 0.0015          # 买入费率
COST_SELL = 0.0025         # 卖出费率
MIN_COMMISSION = 5         # 最低佣金门槛（致命刺客）
# ================= 2. 小资金选股与按手分配权重 =================
def calculate_small_cap_weights(predictions_df, price_df):
    """
    输入: 
    - predictions_df: ['date', 'ticker', 'predict']
    - price_df: ['date', 'ticker', 'close_price'] (用于计算能否买得起1手)
    输出: ['date', 'ticker', 'weight', 'shares'(手数), 'actual_cost'(实际占用资金)]
    """
    def assign_weight_realistic(group, prices):
        # 1. 计算 Rank 百分位
        group['rank_pct'] = group['predict'].rank(pct=True)
        # 2. 筛选 Top N
        top_n = group.nlargest(TOP_N, 'rank_pct').copy()
        # 3. 剔除买不起1手的股票 (核心改造！)
        # 获取当日这些股票的收盘价
        current_prices = prices[prices['date'] == group['date'].iloc[0]].set_index('ticker')['close_price']
        top_n['price'] = top_n['ticker'].map(current_prices)
        # 假设平均分配资金，算出单只股票预算
        budget_per_stock = TOTAL_CAPITAL / TOP_N
        # 买1手需要的资金
        top_n['cost_per_lot'] = top_n['price'] * 100 
        # 剔除买不起的（1手价格 > 单只预算）
        top_n = top_n[top_n['cost_per_lot'] <= budget_per_stock].copy()
        if len(top_n) == 0:
            return pd.DataFrame() # 当天没有买得起的股票
        # 4. 计算理论权重 (指数加权)
        top_n['raw_weight'] = np.exp(LAMBDA * top_n['rank_pct'])
        top_n['theory_weight'] = top_n['raw_weight'] / top_n['raw_weight'].sum()
        # 5. 理论权重转为理论分配资金 -> 转为应买手数 (向下取整)
        top_n['theory_capital'] = top_n['theory_weight'] * TOTAL_CAPITAL
        top_n['lots'] = (top_n['theory_capital'] / top_n['cost_per_lot']).apply(np.floor).astype(int)
        # 防止出现算出来0手的情况，得分最高的至少买1手
        top_n = top_n.sort_values('rank_pct', ascending=False)
        if top_n.iloc[0]['lots'] == 0:
            top_n.iloc[0, top_n.columns.get_loc('lots')] = 1
        # 6. 计算实际占用的资金和实际权重
        top_n['actual_cost'] = top_n['lots'] * top_n['cost_per_lot']
        total_actual_cost = top_n['actual_cost'].sum()
        # 实际权重 = 实际占用资金 / 总资金 (剩余资金视为现金仓位)
        top_n['weight'] = top_n['actual_cost'] / TOTAL_CAPITAL
        return top_n[['ticker', 'weight', 'lots', 'actual_cost']]
    print("正在计算小资金按手分配权重...")
    portfolio = predictions_df.groupby('date').apply(lambda x: assign_weight_realistic(x, price_df)).reset_index()
    portfolio = portfolio.drop(columns=['level_1'])
    return portfolio
# ================= 3. 考虑最低佣金的小资金回测 =================
def run_small_cap_backtest(portfolio_weights, returns_df):
    print("正在运行小资金回测（含最低佣金惩罚）...")
    merged = pd.merge(portfolio_weights, returns_df, on=['date', 'ticker'], how='left')
    # 计算策略日收益
    daily_pnl = merged.groupby('date').apply(lambda x: np.sum(x['weight'] * x['return'])).reset_index()
    daily_pnl.columns = ['date', 'strategy_return']
    # 惩罚最低佣金 (极度简化版：假设每次换仓都是全额换手)
    # 单只股票换仓成本 = max(实际占用资金 * 费率, 5元) / 总资金
    # 这里简化为：只要发生换仓，就扣除一个固定的惩罚性成本
    # 实战中需要精确计算，但这里为了让你意识到5元的威力，先粗略扣除
    daily_pnl['cost'] = 0.0 
    # ... (此处省略复杂的每日清算逻辑，实盘必须用真实成交清算)
    daily_pnl['net_return'] = daily_pnl['strategy_return'] - daily_pnl['cost']
    daily_pnl['nav'] = (1 + daily_pnl['net_return']).cumprod()
    return daily_pnl
# ================= 4. 执行与可视化 =================
# 假设你已经有 OOT 期间的预测结果 oot_predictions 和真实收益 oot_returns
# oot_predictions 格式: DataFrame['date', 'ticker', 'predict']
# oot_returns 格式: DataFrame['date', 'ticker', 'return'] (这里的return必须是绝对的日/周/月收益率，用来算净值)
# --- 模拟调用 ---
# portfolio = calculate_weights(oot_predictions)
# nav_curve = run_backtest(portfolio, oot_returns)
# def plot_nav(nav_df):
#     plt.figure(figsize=(12, 6))
#     plt.plot(nav_df['date'], nav_df['nav'], label='Strategy NAV', color='blue')
#     # 如果有基准（比如沪深300），也可以画上去
#     plt.title('OOT Backtest NAV Curve')
#     plt.xlabel('Date')
#     plt.ylabel('Net Asset Value')
#     plt.legend()
#     plt.grid(True)
#     plt.show()
# plot_nav(nav_curve)